## Benchmark 1: Custom Functions

In [ ]:
from jaxkan.models.KAN import KAN
from jaxkan.grids import (
    AdaptationState,
    update_state,
    reset_after_adaptation,
    UniformDensity,
    CurvatureDensity,
    MixedAdaptation,
    ScheduledTrigger
)

import jax
import jax.numpy as jnp
from flax import nnx
import optax

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import json

import os
import time
from datetime import datetime
from pathlib import Path

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

## Load Metadata and Data

In [ ]:
# Load metadata
metadata_path = Path('benchmarks/custom_funcs_data/metadata.json')
with open(metadata_path, 'r') as f:
    CUSTOM_METADATA = json.load(f)


def load_custom_data(func_id):
    
    data_dir = Path('benchmarks/custom_funcs_data')
    filename = data_dir / f"{func_id}.npz"
    
    if not filename.exists():
        raise FileNotFoundError(f"Dataset not found: {filename}")
    
    data = np.load(filename, allow_pickle=True)
    metadata = CUSTOM_METADATA[func_id]
    
    return {
        'X': jnp.array(data['X']),
        'y': jnp.array(data['y']),
        'bounds': metadata['bounds'],
        'n_in': metadata['n_in']
    }

## Evaluation Function

In [ ]:
# Import true functions for evaluation
from benchmarks.custom_funcs import get_function_info

def compute_relative_l2_error(model, func, bounds, n_eval=None, seed=None):
    
    n_in = len(bounds)
    
    # Create structured grid based on dimensionality
    if n_in == 1:
        # 1000 points for 1D
        n_points = 1000
        x0 = jnp.linspace(bounds[0][0], bounds[0][1], n_points)
        X_eval = x0.reshape(-1, 1)
    elif n_in == 2:
        # 200×200 grid for 2D
        n_points = 200
        x0 = jnp.linspace(bounds[0][0], bounds[0][1], n_points)
        x1 = jnp.linspace(bounds[1][0], bounds[1][1], n_points)
        X0, X1 = jnp.meshgrid(x0, x1, indexing='ij')
        X_eval = jnp.stack([X0.ravel(), X1.ravel()], axis=1)
    elif n_in == 4:
        # 20^4 grid for 4D (160,000 points)
        n_points = 20
        grids = [jnp.linspace(bounds[i][0], bounds[i][1], n_points) for i in range(4)]
        mesh = jnp.meshgrid(*grids, indexing='ij')
        X_eval = jnp.stack([m.ravel() for m in mesh], axis=1)
    elif n_in == 6:
        # 10^6 grid for 6D (1,000,000 points)
        n_points = 10
        grids = [jnp.linspace(bounds[i][0], bounds[i][1], n_points) for i in range(6)]
        mesh = jnp.meshgrid(*grids, indexing='ij')
        X_eval = jnp.stack([m.ravel() for m in mesh], axis=1)
    else:
        raise ValueError(f"Unsupported dimensionality: {n_in}D")
    
    # Evaluate true function
    y_true = func(X_eval)
    
    # Compute predictions
    y_pred = model(X_eval)
    
    # Compute L^2 norms
    error_norm = jnp.sqrt(jnp.mean((y_pred - y_true) ** 2))
    true_norm = jnp.sqrt(jnp.mean(y_true ** 2))
    
    # Relative L^2 error
    rel_l2_error = float(error_norm / true_norm)
    
    return rel_l2_error

## Training Functions

In [ ]:
@nnx.jit
def train_step(model, optimizer, X_batch, y_batch):
    
    def loss_fn(model):
        residual = model(X_batch) - y_batch
        loss = jnp.mean(residual**2)
        return loss
    
    loss, grads = nnx.value_and_grad(loss_fn)(model)
    optimizer.update(model, grads)
    
    return loss


@nnx.jit
def compute_curvature(model, X):
    
    epsilon=1e-3
    n_in = X.shape[1]
    curvatures = jnp.zeros(X.shape[0])
    
    for dim in range(n_in):
        h = jnp.zeros((1, n_in))
        h = h.at[0, dim].set(epsilon)
        
        X_plus = X + h
        X_minus = X - h
        
        f_center = model(X)
        f_plus = model(X_plus)
        f_minus = model(X_minus)
        
        second_deriv = (f_plus - 2 * f_center + f_minus) / (epsilon ** 2)
        curvatures = curvatures + jnp.sum(jnp.abs(second_deriv), axis=1)
    
    return curvatures


def run_training(
    func_id,
    architecture,
    method,
    seed,
    num_epochs=3000,
    learning_rate=0.01,
    grid_schedule=None,
    verbose=False
):
    
    start_time = time.time()
    
    # Load pre-generated training data
    data = load_custom_data(func_id)
    X_train = data['X']
    y_train = data['y']
    bounds = data['bounds']
    n_in = data['n_in']
    
    # Get true function for evaluation
    func_info = get_function_info(func_id)
    func = func_info['function']
    
    # Create model
    req_params = {'k': 3, 'G': 3, 'init_scheme': {'type': 'glorot_fine'}}
    model = KAN(
        layer_dims=architecture,
        layer_type='spline',
        required_parameters=req_params,
        seed=seed
    )
    
    optimizer = nnx.Optimizer(model, optax.adam(learning_rate), wrt=nnx.Param)
    
    # Setup adaptation framework
    state = AdaptationState()
    trigger = ScheduledTrigger(epochs=list(grid_schedule.keys()))
    
    # Configure IDF and strategy based on method
    if method == 'input_adaptive':
        idf = UniformDensity()
        strategy = MixedAdaptation(grid_e=0.0)  # Quantile-based on input distribution
    elif method == 'curvature_adaptive':
        idf = CurvatureDensity()
        strategy = MixedAdaptation(grid_e=0.0)  # Quantile-based on curvature
    else:
        raise ValueError(f"Unknown method: {method}")
    
    # Training loop
    train_losses = []
    
    for epoch in range(num_epochs):
        # Check for grid adaptation
        if trigger(state):
            G_new = grid_schedule[epoch]
            
            if verbose:
                print(f"  Epoch {epoch}: Adapting grid to G={G_new}")
            
            # Compute curvatures if needed
            if method == 'curvature_adaptive':
                curvatures = compute_curvature(model, X_train)
                model.update_grids(
                    X_train,
                    idf=idf,
                    strategy=strategy,
                    grid_size_new=G_new,
                    curvatures=curvatures
                )
            else:
                model.update_grids(
                    X_train,
                    idf=idf,
                    strategy=strategy,
                    grid_size_new=G_new
                )
            
            # Reset optimizer after grid change
            optimizer = nnx.Optimizer(model, optax.adam(learning_rate), wrt=nnx.Param)
            state = reset_after_adaptation(state)
        
        # Training step
        loss = train_step(model, optimizer, X_train, y_train)
        train_losses.append(float(loss))
        
        state = update_state(state, loss=loss)
    
    # Final evaluation on structured grid
    rel_l2_error = compute_relative_l2_error(model, func, bounds)
    
    wall_time = time.time() - start_time
    
    if verbose:
        print(f"  Relative L^2 error: {rel_l2_error:.6e} (time: {wall_time:.1f}s)")
    
    return {
        'rel_l2_error': rel_l2_error,
        'train_losses': np.array(train_losses),
        'wall_time': wall_time
    }

## Experiment Configuration

In [ ]:
# Selected functions for benchmark
BENCHMARK_FUNCTIONS = ['f1', 'f2', 'f3', 'f4', 'f5', 'f6', 'f7', 'f8', 'f9', 'f10']

# Architecture configuration
# Note: n_in will be set dynamically based on function
ARCHITECTURE = lambda n_in: [n_in, 10, 1]

# Methods to compare
METHODS = ['input_adaptive', 'curvature_adaptive']

# Number of random seeds
N_SEEDS = 5
SEEDS = list(range(42, 42 + N_SEEDS))

# Training configuration
TRAINING_CONFIG = {
    'num_epochs': 2000,
    'learning_rate': 0.01,
    'grid_schedule': {0: 3, 500: 6, 1000: 9, 1500: 12}
}

# Print configuration summary
print("Benchmark Configuration:")
print(f"  Functions: {BENCHMARK_FUNCTIONS}")
print(f"  Methods: {METHODS}")
print(f"  Seeds: {N_SEEDS}")
print(f"\nTraining Config:")
print(f"  Epochs: {TRAINING_CONFIG['num_epochs']}")
print(f"  Grid schedule: {TRAINING_CONFIG['grid_schedule']}")
print(f"\nTotal experiments: {len(BENCHMARK_FUNCTIONS)} × {len(METHODS)} × {N_SEEDS}")
print(f"  = {len(BENCHMARK_FUNCTIONS) * len(METHODS) * N_SEEDS} training runs")

## Run Benchmark


In [ ]:
# Storage for results
results = []

total_runs = len(BENCHMARK_FUNCTIONS) * len(METHODS) * N_SEEDS
current_run = 0

print(f"Starting benchmark: {total_runs} total runs")
print(f"Started at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("=" * 80)

for func_id in BENCHMARK_FUNCTIONS:
    func_info = CUSTOM_METADATA[func_id]
    print(f"\nFunction: {func_id} - {func_info['name']} ({func_info['n_in']}D)")
    
    architecture = ARCHITECTURE(func_info['n_in'])
        
    for method in METHODS:
        print(f"      Method: {method}", end=" ")
        
        method_results = []
        
        for seed in SEEDS:
            current_run += 1
            
            result = run_training(
                func_id=func_id,
                architecture=architecture,
                method=method,
                seed=seed,
                **TRAINING_CONFIG,
                verbose=False
            )
            
            # Store result
            results.append({
                'func_id': func_id,
                'n_in': func_info['n_in'],
                'arch_str': str(architecture),
                'method': method,
                'seed': seed,
                'rel_l2_error': result['rel_l2_error'],
                'wall_time': result['wall_time']
            })
            
            method_results.append(result['rel_l2_error'])
        
        # Print summary for this method
        med_error = np.median(method_results)
        std_error = np.std(method_results)
        print(f"→ Med L^2: {med_error:.3e} ± {std_error:.3e} \t [{current_run}/{total_runs}]")

print("\n" + "=" * 80)
print(f"Benchmark complete at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Total runs: {len(results)}")

## Save Results

In [ ]:
# Convert to DataFrame
df_results = pd.DataFrame(results)

# Save to CSV
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
filename = f'results/custom_results_{timestamp}.csv'
df_results.to_csv(filename, index=False)
print(f"Results saved to: {filename}")
